# 01_14 — Exemplo de código com apoio de LLM

> **Material de familiarização com o JupyterLab do SimServ.**  
> Estes notebooks foram pensados para ficar em uma pasta de somente leitura, por exemplo `/sobre_ambientes_simserv`.  
> Para editar, testar livremente ou salvar resultados, faça uma cópia para sua área de trabalho, como `/home/jovyan/ambiente_laboratorio`.

## Objetivo

Mostrar um exemplo simples de como um modelo de linguagem pode explicar um trecho de código.

Este notebook tenta usar a IA local se ela estiver disponível. Se não estiver, o notebook continua funcionando e mostra uma explicação escrita no próprio material.

In [1]:
# 1. Defina o valor da variável (usando aspas para string)
codigo_exemplo = "   Exemplo de código com espaços   "

for i in range(5):
    # 2. Primeiro você trata a string com .strip(), depois imprime
    # (Neste exemplo, usei o i apenas para ilustrar o loop)
    print(f"Linha {i}: {codigo_exemplo.strip()}")

print("\nCódigo que será explicado:")
print(codigo_exemplo)

Linha 0: Exemplo de código com espaços
Linha 1: Exemplo de código com espaços
Linha 2: Exemplo de código com espaços
Linha 3: Exemplo de código com espaços
Linha 4: Exemplo de código com espaços

Código que será explicado:
   Exemplo de código com espaços   


In [9]:
import json
import socket
from urllib.request import urlopen, Request

def carregar_modelo_na_memoria(modelo):
    payload = {"model": modelo}
    req = Request(
        "http://ollama:11434/api/generate", # Endpoint de geração
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )
    try:
        # Enviamos o pedido e não esperamos a resposta completa (apenas o sinal de recebido)
        urlopen(req, timeout=5) 
    except Exception:
        print(f"Modelo {modelo} está sendo carregado em segundo plano...")

# Use antes de consultar
carregar_modelo_na_memoria("qwen2.5-coder:32b")

In [10]:
def listar_modelos_ollama():
    import socket
    socket.setdefaulttimeout(900) # Força o socket a ter paciência
    
    for base in ["http://ollama:11434"]:
        try:
            # Adicionamos headers explicitamente para manter a conexão viva
            req = Request(base + "/api/tags", headers={"Connection": "keep-alive"})
            with urlopen(req, timeout=900) as resposta:
                dados = json.loads(resposta.read().decode("utf-8"))
            return base, [m.get("name") for m in dados.get("models", [])]
        except Exception as e:
            print(f"Tentativa falhou: {e}")
    return None, []

In [4]:
listar_modelos_ollama()

('http://ollama:11434',
 ['deepseek-coder:33b',
  'qwen2.5-coder:32b',
  'qwen2.5-coder:14b',
  'deepseek-coder:6.7b'])

In [5]:
import json
from urllib.request import urlopen, Request

def listar_modelos_ollama():
    for base in ["http://ollama:11434/api/generate", "http://ollama:11434", "http://127.0.0.1:11434", "http://localhost:11434", "http://ollama:11434/api/tags"]:
        try:
            with urlopen(base + "/api/tags", timeout=900) as resposta:
                dados = json.loads(resposta.read().decode("utf-8"))
            modelos = [m.get("name") for m in dados.get("models", []) if m.get("name")]
            return base, modelos
        except Exception:
            pass
    return None, []

In [6]:
listar_modelos_ollama()

('http://ollama:11434',
 ['deepseek-coder:33b',
  'qwen2.5-coder:32b',
  'qwen2.5-coder:14b',
  'deepseek-coder:6.7b'])

In [7]:
def consultar_ollama(prompt, modelo=None):
    base, modelos = listar_modelos_ollama()
    if not base or not modelos:
        return None
    modelo = modelo or modelos[0]
    payload = {
        "model": modelo,
        "prompt": prompt,
        "stream": False,
    }
    req = Request(
        base + "/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urlopen(req, timeout=900) as resposta:
        dados = json.loads(resposta.read().decode("utf-8"))
    return dados.get("response")

In [8]:
# 1. Definição do Prompt (usando f-string e strip para limpar espaços)
codigo_exemplo = """
def extrair_autores(texto):
    linhas = texto.split("\\n")[:30]
    autores = []
    for linha in linhas:
        if "universidade" in linha.lower():
            break
        if len(linha.split()) <= 6 and any(p[0].isupper() for p in linha.split()):
            autores.append(linha.strip())
    return "; ".join(dict.fromkeys(autores)) if autores else "N/A"
"""

prompt = f"""
Explique o código Python abaixo para um iniciante.
Seja claro, breve e didático.

Código:
{codigo_exemplo}
""".strip()

# 2. Chamada da IA Local
resposta = consultar_ollama(prompt)

# 3. Lógica de exibição da resposta
if resposta:
    # Use \n entre aspas para pular linha corretamente
    print("Resposta da IA local:\n") 
    print(resposta)
else:
    print("A IA local não respondeu neste momento.\n")
    print("Explicação alternativa:")
    print("O comando 'for' repete uma ação. Neste exemplo, a variável 'i' assume os valores 0, 1, 2, 3 e 4.")
    print("A função print(i) mostra cada valor na tela.")

Resposta da IA local:

Este código Python é usado para extrair os nomes dos autores de um texto. Ele funciona da seguinte maneira:

1. A função `extrair_autores()` recebe como parâmetro o próprio texto.
2. Ela divide este texto em linhas com base no caractere de nova linha `\n` e seleciona apenas as primeiras 30 dessas linhas.
3. Cria-se um array vazio chamado `autores` para armazenar os nomes dos autores.
4. Então, para cada linha da lista de linhas:
   - Se a palavra "universidade" for encontrada na linha (ignorando se ela estiver em maiúscula ou minúscula), o loop é interrompido porque é provável que tenhamos chegado à parte dos autores do texto.
   - Se a quantidade de palavras da linha for menor ou igual a 6 e pelo menos uma dessas palavras começar com letra maiúscula, a linha é considerada um nome de autor potencial. A linha é adicionada à lista `autores`.
5. No final, se a lista `autores` não estiver vazia, ele retorna todos os nomes únicos dos autores separados por ";". Se esti

In [8]:
# 1. Armazene o código que você quer explicar em uma variável de texto (string)
codigo_exemplo = """
def extrair_autores(texto):
    linhas = texto.split("\\n")[:30]
    autores = []
    for linha in linhas:
        if "universidade" in linha.lower():
            break
        if len(linha.split()) <= 6 and any(p[0].isupper() for p in linha.split()):
            autores.append(linha.strip())
    return "; ".join(dict.fromkeys(autores)) if autores else "N/A"
"""

# 2. Definição do Prompt (usando f-string)
prompt = f"""
Explique o código Python abaixo para um iniciante.
Seja claro, breve e didático.

Código:
{codigo_exemplo}
""".strip()

print(prompt)

Explique o código Python abaixo para um iniciante.
Seja claro, breve e didático.

Código:

def extrair_autores(texto):
    linhas = texto.split("\n")[:30]
    autores = []
    for linha in linhas:
        if "universidade" in linha.lower():
            break
        if len(linha.split()) <= 6 and any(p[0].isupper() for p in linha.split()):
            autores.append(linha.strip())
    return "; ".join(dict.fromkeys(autores)) if autores else "N/A"


In [9]:
import json
from urllib.request import urlopen, Request

   
# 1. Armazene o código que você quer explicar
codigo_exemplo = """
def extrair_autores(texto):
    linhas = texto.split("\\n")[:30]
    autores = []
    for linha in linhas:
        if "universidade" in linha.lower():
            break
        if len(linha.split()) <= 6 and any(p[0].isupper() for p in linha.split()):
            autores.append(linha.strip())
    return "; ".join(dict.fromkeys(autores)) if autores else "N/A"
"""

# 2. Definição do Prompt para a IA
prompt = f"""
Explique o código Python abaixo para um iniciante.
Seja claro, breve e didático.

Código:
{codigo_exemplo}
""".strip()

# 3. Funções de comunicação com o Ollama Interno
def listar_modelos_ollama():
    # Tenta conectar em diferentes endereços comuns
    for base in ["http://ollama:11434", "http://127.0.0.1:11434", "http://localhost:11434"]:
        try:
            with urlopen(base + "/api/tags", timeout=600) as resposta:
                dados = json.loads(resposta.read().decode("utf-8"))
            modelos = [m.get("name") for m in dados.get("models", []) if m.get("name")]
            if modelos:
                return base, modelos
        except Exception:
            pass
    return None, []

def consultar_ollama(prompt, modelo=None):
    base, modelos = listar_modelos_ollama()
    if not base or not modelos:
        print("Erro: Ollama não encontrado ou nenhum modelo baixado.")
        return None
    
    # Seleciona o primeiro modelo disponível se nenhum for especificado (ex: llama3, mistral)
    modelo_escolhido = modelo or modelos[0]
    
    payload = {
        "model": modelo_escolhido,
        "prompt": prompt,
        "stream": False,
    }
    
    try:
        req = Request(
            base + "/api/generate",
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"},
        )
        with urlopen(req, timeout=600) as resposta:
            dados = json.loads(resposta.read().decode("utf-8"))
        return dados.get("response")
    except Exception as e:
        print(f"Erro na consulta: {e}")
        return None

# 4. Execução e Resposta
print(f"--- Iniciando consulta ao Ollama ---")
resposta_ia = consultar_ollama(prompt)

if resposta_ia:
    print("\nEXPLICAÇÃO DA IA:\n")
    print(resposta_ia)
else:
    print("\nNão foi possível obter uma resposta da IA local.")

--- Iniciando consulta ao Ollama ---



EXPLICAÇÃO DA IA:

Esse código Python é usado para extrair os primeiros nomes dos autores de um texto. O objetivo é detectar se uma frase está escrita em maiúscula e possui no máximo seis palavras, o que normalmente indica que esta é a linha que contém o nome do autor. A partir desse critério, o código extrai as primeiras 30 linhas do texto para detectar os autores.

A seguir é explicado passo-a-passo:

1. `linhas = texto.split("\n")[:30]`: O texto recebido é dividido em linhas e são consideradas apenas as primeiras 30 (slicing).

2. `autores = []`: Inicialmente, a lista de autores está vazia. Os nomes dos autores serão adicionados nessa lista.

3. `for linha in linhas:`: O código percorre cada uma das primeiras 30 linhas do texto.

4. `if "universidade" in linha.lower(): break`: Se a palavra 'universidade' aparece em qualquer linha minúscula ou maiúscula, o loop deve ser interrompido imediatamente porque geralmente essa é a última linha dos autores no texto.

5. `if len(linha.split()

In [10]:
prompt_melhoria = """
Analise minha função Python de extração de autores e sugira uma melhoria 
usando Expressões Regulares (Regex) para capturar nomes que tenham 
abreviações (ex: 'Silva, J. B.').

Código atual:
def extrair_autores(texto):
    linhas = texto.split("\\n")[:30]
    autores = []
    for linha in linhas:
        if "universidade" in linha.lower(): break
        if len(linha.split()) <= 6 and any(p[0].isupper() for p in linha.split()):
            autores.append(linha.strip())
    return "; ".join(dict.fromkeys(autores))
"""

print(consultar_ollama(prompt_melhoria, modelo="qwen2.5-coder:32b"))

Seu código atual já faz um bom trabalho para extrair nomes de autores, mas pode ser aprimorado para lidar com abreviações usando expressões regulares (Regex). Aqui está uma versão melhorada da sua função:

```python
import re

def extrair_autores(texto):
    linhas = texto.split("\n")[:30]
    autores = []
    
    # Definindo um padrão regex para nomes de autor que podem incluir abreviações
    padrao_autor = r'^[A-Za-zÀ-ú]+(?:, [A-Z]\. ?)+$'
    
    for linha in linhas:
        if "universidade" in linha.lower(): break
        # Usando regex para verificar se a linha corresponde ao padrão de nome de autor
        if re.match(padrao_autor, linha.strip()):
            autores.append(linha.strip())
            
    return "; ".join(dict.fromkeys(autores))

# Exemplo de uso
texto = """Silva, J. B.
Pereira, Maria C.
Oliveira, Ana M.; Souza, L. F.
Fernandes, Pedro A.; Costa, R. S.
Universidade Federal do Rio de Janeiro"""
print(extrair_autores(texto))
```

### Explicação dos Ajustes:

1. 

## Atividade

Modifique o código de exemplo para imprimir os números de 0 a 9. Depois peça à IA para explicar a diferença.